In [ ]:
import gmsh

gmsh.initialize()
gmsh.model.add("cross_junction")
geo = gmsh.model.geo

# ---------------- Dimensions [m] ----------------
H   = 0.01    # channel width / height
L_in  = 0.02  # main channel length upstream of junction
L_out = 0.05  # main channel length downstream of junction
L_v   = 0.02  # length of top / bottom branches
lc  = 0.0004  # global mesh size (~25 elements across a channel)
lc_j = 0.0002 # finer size near the junction

x0, x1, x2 = 0.0, L_in, L_in + H          # 0, 0.02, 0.03
x3 = L_in + H + L_out                      # 0.08
y0, y1 = 0.0, H                            # main channel: 0 -> 0.01
yb, yt = -L_v, H + L_v                     # bottom / top ends: -0.02 / 0.03

# ---------------- Points (counter-clockwise) ----------------
pts = [
    (x0, y0), (x1, y0),   # 1, 2   bottom wall of left channel
    (x1, yb), (x2, yb),   # 3, 4   bottom branch end (inlet)
    (x2, y0), (x3, y0),   # 5, 6   bottom wall of right channel
    (x3, y1), (x2, y1),   # 7, 8   outlet top / right channel top wall
    (x2, yt), (x1, yt),   # 9, 10  top branch end (inlet)
    (x1, y1), (x0, y1),   # 11, 12 top wall of left channel
]
# smaller mesh size on the four junction corners (points 2, 5, 8, 11)
junction_ids = {2, 5, 8, 11}
p = []
for i, (x, y) in enumerate(pts, start=1):
    size = lc_j if i in junction_ids else lc
    p.append(geo.addPoint(x, y, 0, size))

# ---------------- Lines ----------------
n = len(p)
l = [geo.addLine(p[i], p[(i + 1) % n]) for i in range(n)]
# l[0]  p1-p2    wall
# l[1]  p2-p3    wall
# l[2]  p3-p4    INLET bottom
# l[3]  p4-p5    wall
# l[4]  p5-p6    wall
# l[5]  p6-p7    OUTLET
# l[6]  p7-p8    wall
# l[7]  p8-p9    wall
# l[8]  p9-p10   INLET top
# l[9]  p10-p11  wall
# l[10] p11-p12  wall
# l[11] p12-p1   INLET left

loop = geo.addCurveLoop(l)
surf = geo.addPlaneSurface([loop])
geo.synchronize()

# ---------------- Physical groups ----------------
inlet_left   = gmsh.model.addPhysicalGroup(1, [l[11]], name="inlet_left")
inlet_top    = gmsh.model.addPhysicalGroup(1, [l[8]],  name="inlet_top")
inlet_bottom = gmsh.model.addPhysicalGroup(1, [l[2]],  name="inlet_bottom")
outlet       = gmsh.model.addPhysicalGroup(1, [l[5]],  name="outlet")

wall_lines = [l[0], l[1], l[3], l[4], l[6], l[7], l[9], l[10]]
walls = gmsh.model.addPhysicalGroup(1, wall_lines, name="walls")  # no-slip

fluid = gmsh.model.addPhysicalGroup(2, [surf], name="water")

# ---------------- Mesh ----------------
# Optional: quad-dominant mesh (comment out for triangles)
gmsh.option.setNumber("Mesh.RecombineAll", 1)
gmsh.option.setNumber("Mesh.Algorithm", 8)  # Frontal-Delaunay for quads

gmsh.model.mesh.generate(2)


from pathlib import Path
# write the .msh to same folder
mesh_file = Path("cross_junction.msh")

gmsh.write(str(mesh_file))

gmsh.fltk.run()
gmsh.finalize()
